<a href="https://colab.research.google.com/github/chitta-behera/Machine-Learning/blob/master/centimate_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
reviews = [

    "This movie is amazing",

    "I really loved this film",

    "Fantastic acting and story",

    "Wonderful experience",

    "Best movie ever",

    "This movie is terrible",

    "Worst film I have seen",

    "Very boring movie",

    "I hate this movie",

    "Bad acting"
]

labels = [

    1,
    1,
    1,
    1,
    1,

    0,
    0,
    0,
    0,
    0
]

In [3]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [4]:
reviews = [

    clean_text(review)

    for review in reviews
]

In [5]:
tokenized = []

for review in reviews:

    tokenized.append(

        review.split()

    )

print(tokenized)

[['this', 'movie', 'is', 'amazing'], ['i', 'really', 'loved', 'this', 'film'], ['fantastic', 'acting', 'and', 'story'], ['wonderful', 'experience'], ['best', 'movie', 'ever'], ['this', 'movie', 'is', 'terrible'], ['worst', 'film', 'i', 'have', 'seen'], ['very', 'boring', 'movie'], ['i', 'hate', 'this', 'movie'], ['bad', 'acting']]


In [6]:
vocab = {}

index = 1

for sentence in tokenized:

    for word in sentence:

        if word not in vocab:

            vocab[word] = index

            index += 1

print(vocab)

{'this': 1, 'movie': 2, 'is': 3, 'amazing': 4, 'i': 5, 'really': 6, 'loved': 7, 'film': 8, 'fantastic': 9, 'acting': 10, 'and': 11, 'story': 12, 'wonderful': 13, 'experience': 14, 'best': 15, 'ever': 16, 'terrible': 17, 'worst': 18, 'have': 19, 'seen': 20, 'very': 21, 'boring': 22, 'hate': 23, 'bad': 24}


In [7]:
encoded = []

for sentence in tokenized:

    temp = []

    for word in sentence:

        temp.append(

            vocab[word]

        )

    encoded.append(temp)

print(encoded)

[[1, 2, 3, 4], [5, 6, 7, 1, 8], [9, 10, 11, 12], [13, 14], [15, 2, 16], [1, 2, 3, 17], [18, 8, 5, 19, 20], [21, 22, 2], [5, 23, 1, 2], [24, 10]]


In [8]:
max_len = max(

    len(sentence)

    for sentence in encoded
)

In [9]:
padded = []

for sentence in encoded:

    while len(sentence) < max_len:

        sentence.append(0)

    padded.append(sentence)

print(padded)

[[1, 2, 3, 4, 0], [5, 6, 7, 1, 8], [9, 10, 11, 12, 0], [13, 14, 0, 0, 0], [15, 2, 16, 0, 0], [1, 2, 3, 17, 0], [18, 8, 5, 19, 20], [21, 22, 2, 0, 0], [5, 23, 1, 2, 0], [24, 10, 0, 0, 0]]


In [10]:
X = torch.LongTensor(padded)

y = torch.FloatTensor(labels).view(-1,1)

In [11]:
torch.LongTensor()

tensor([], dtype=torch.int64)

In [12]:
class ReviewDataset(Dataset):

    def __init__(self,X,y):

        self.X = X

        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self,index):

        return self.X[index],self.y[index]

In [13]:
dataset = ReviewDataset(X,y)

loader = DataLoader(

    dataset,

    batch_size=2,

    shuffle=True
)

In [14]:
class SentimentModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(

            num_embeddings=len(vocab)+1,

            embedding_dim=8
        )

        self.fc = nn.Sequential(

            nn.Linear(8,16),

            nn.ReLU(),

            nn.Linear(16,8),

            nn.ReLU(),

            nn.Linear(8,1)
        )

    def forward(self,x):

        x = self.embedding(x)

        x = x.mean(dim=1)

        x = self.fc(x)

        return x

In [15]:
model = SentimentModel()

In [16]:
criterion = nn.BCEWithLogitsLoss()

In [17]:
optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.01
)

In [18]:
epochs = 50

for epoch in range(epochs):

    for X_batch,y_batch in loader:

        outputs = model(X_batch)

        loss = criterion(

            outputs,

            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    if epoch % 10 == 0:

        print(

            f"Epoch {epoch} Loss {loss.item():.4f}"

        )

Epoch 0 Loss 0.6810
Epoch 10 Loss 0.4255
Epoch 20 Loss 0.0039
Epoch 30 Loss 0.0001
Epoch 40 Loss 0.0017


In [19]:
test = "This movie is fantastic"

In [20]:
test = clean_text(test)
words = test.split()

In [21]:
encoded = []

for word in words:

    encoded.append(

        vocab.get(word,0)

    )

In [22]:
while len(encoded) < max_len:

    encoded.append(0)

In [23]:
test = torch.LongTensor([encoded])

In [24]:
model.eval()

with torch.no_grad():

    output = model(test)

    probability = torch.sigmoid(output)

    print(probability)

    if probability.item() > 0.5:

        print("Positive Review")

    else:

        print("Negative Review")

tensor([[0.9996]])
Positive Review
